**Version 2 — fixes retrieval relevance and latency. Open this file in a fresh kernel and Run All; no index rebuild is required.**

# Wikipedia RAG — Question & Answer UI
Run **all cells once**, then type questions into the interface at the bottom. This notebook reuses the completed full Wikipedia BM25 index and local Flan-T5-small model. It does not build an index, download assets, or run ablations.

Each click retrieves real passages from SQLite, sends that evidence through Flan-T5-small, and shows the answer, supporting citations, and all retrieved passages. Concatenate is the faster default. Generate-from-evidence makes up to k+1 model calls. Retrieval has a 15-second SQL budget and caches repeated questions. Generation latency depends on hardware. Historical questions should include the period or year. The evidence ranking is heuristic, not a semantic entailment verifier. Wait for the status to change to complete before submitting another question.

Use the same Python kernel as the full notebook. If widgets are missing, install `ipywidgets` in that kernel while online (`%pip install ipywidgets`), restart the kernel, and rerun. JupyterLab, Notebook and VS Code must have widget rendering enabled. After a kernel restart, run all cells again. No API key is required.


**Updated retrieval:** conjunctive native FTS ranking; article introduction expansion; sentence-level relevance and navigation/historical penalties. Only short model answers are accepted. No facts are hardcoded. This UI retrieval differs from the original ablation baseline, whose recorded results remain unchanged.

Retrieval size is an upper bound: weak candidates more than four heuristic score points below the best evidence are dropped. The UI may use fewer than k passages to avoid mixing a direct count with counts of subsets.


In [1]:
# Load Arrow first to match the working full-notebook runtime on this environment.
import pyarrow
from pathlib import Path
import os, re, json, sqlite3, time, html
from urllib.parse import urlparse
from IPython.display import display, HTML, clear_output
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("Install ipywidgets in this kernel, restart, then rerun this notebook.") from exc

ROOT = Path('/Users/amolsingh/Documents/code/IITB-DL')
INDEX_PATH = ROOT / 'data/full_wikipedia_bm25.sqlite'
MODEL_DIR = ROOT / 'models/flan-t5-small'
DEVICE = 'cpu'
THREADS = 4


In [2]:
def offline_mode() -> None:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [3]:
UNKNOWN = "I don't know based on the provided evidence."
WORD = re.compile(r"\w+", re.UNICODE)
STOPWORDS = set(
    "a an the and or but of to in on at for from by with as is are was were be "
    "been being do does did has have had what which who whom whose when where "
    "why how can could would should will may might that this these those it its "
    "into about according many much name tell me give".split()
)


In [4]:
def span(text: str, candidate: str) -> str | None:
    """Return an original-text span, matching complete casefolded word tokens."""
    needles = [m.group().casefold() for m in WORD.finditer(candidate)]
    if not needles:
        return None
    matches = list(WORD.finditer(text))
    haystack = [m.group().casefold() for m in matches]
    for i in range(len(haystack) - len(needles) + 1):
        if haystack[i : i + len(needles)] == needles:
            return text[matches[i].start() : matches[i + len(needles) - 1].end()]
    return None

def is_unknown(text: str) -> bool:
    normalized = " ".join(WORD.findall(text.casefold()))
    return not normalized or normalized in {
        "unknown", "unanswerable", "not enough information", "not provided",
        "i don t know", "i do not know", "no answer", "not in the context",
        "i don t know based on the provided evidence",
    }

def passages(article: dict, chunk_words: int = 100, overlap: int = 25):
    if chunk_words < 1 or overlap < 0 or overlap >= chunk_words:
        raise ValueError("Require chunk_words > overlap >= 0.")
    text = article["text"]
    words = list(re.finditer(r"\S+", text))
    for ordinal, start in enumerate(range(0, len(words), chunk_words - overlap)):
        end = min(start + chunk_words, len(words))
        char_start, char_end = words[start].start(), words[end - 1].end()
        yield {
            "passage_id": f'{article["id"]}:{ordinal}',
            "article_id": str(article["id"]),
            "title": article["title"],
            "url": article["url"],
            "text": text[char_start:char_end],
            "char_start": char_start,
            "char_end": char_end,
        }
        if end == len(words):
            break


In [5]:
class Retriever:
    """Bounded conjunctive BM25 retrieval followed by transparent sentence reranking."""
    def __init__(self, path: Path):
        self.connection = sqlite3.connect(path.resolve().as_uri()+'?mode=ro',uri=True)
        self.connection.row_factory = sqlite3.Row
        self.metadata = {r['key']:json.loads(r['value']) for r in self.connection.execute('SELECT key,value FROM metadata')}
        if not self.metadata.get('complete'):
            self.connection.close()
            raise RuntimeError('Full index is incomplete.')
        self.cache = {}
        self.last_diagnostics = {}

    def close(self):
        self.connection.close()

    @staticmethod
    def stem(word):
        word = word.casefold()
        return word[:-1] if word.endswith('s') and not word.endswith('ss') else word

    def search(self, question, k=3):
        import copy
        if not 0 <= k <= 5:
            raise ValueError('Choose k between 0 and 5.')
        if not question.strip():
            raise ValueError('Enter a question.')
        if k == 0:
            return []
        key = question.strip().casefold()
        if key in self.cache:
            self.last_diagnostics = dict(self.cache[key][1],cached=True)
            return copy.deepcopy(self.cache[key][0][:k])
        terms = list(dict.fromkeys(w.casefold() for w in WORD.findall(question)
                     if w.casefold() not in STOPWORDS | {'there','please','currently','current','total','during','before','after','until'}))
        if not terms:
            return []
        if len(terms)>16:
            raise ValueError('Please use a shorter factual question (at most 16 content words).')
        query_terms = {self.stem(t) for t in terms}
        historical = bool(re.search(r'\b(was|were|former|historical|history|before|during|until|\d{4})\b',key))
        count_match = re.search(r'how many\s+(.+?)(?:\s+(?:are|does|do|in|has|have|exist)\b|\?)',key+'?')
        count_noun = count_match.group(1).strip() if count_match else None
        relation_match = re.search(r'(?:what|which)\s+(?:is|was|are|were)\s+(?:the\s+)?(.+?)\s+of\s+(.+?)[?.]*$',key)
        phrase = ' '.join(relation_match.groups()) if relation_match else None
        exact_relation = ' of '.join(relation_match.groups()) if relation_match else None
        if exact_relation:
            exact_relation = re.split(r'\s+(?:during|before|after|until|in \d{4})\b',exact_relation)[0]
        queries = []
        if exact_relation:
            queries.append('text : "'+exact_relation.replace('"','""')+'"')
        queries.append(' AND '.join('"'+t+'"' for t in terms))
        candidates = {}
        start = time.perf_counter()
        deadline = start+15
        self.connection.set_progress_handler(lambda: int(time.perf_counter()>deadline),10000)
        try:
            for query in queries:
                # ORDER BY the native rank column allows FTS5's top-k path. Fetch
                # passage text only after ranking, not a join/sort over all matches.
                rows = self.connection.execute('SELECT rowid FROM search WHERE search MATCH ? ORDER BY rank LIMIT 40',(query,)).fetchall()
                for row in rows:
                    p = dict(self.connection.execute('SELECT * FROM passages WHERE rowid=?',(row[0],)).fetchone())
                    candidates[p['passage_id']] = p
        except sqlite3.OperationalError as exc:
            if 'interrupted' in str(exc):
                raise RuntimeError('Search exceeded 15 seconds. Add a specific name or phrase and retry.') from exc
            raise
        finally:
            self.connection.set_progress_handler(None,0)
        # A matching article footer often outranks its useful introduction. Fetch
        # its first two chunks using the existing unique passage_id index.
        article_ids = list(dict.fromkeys(p['article_id'] for p in candidates.values()))[:50]
        for article_id in article_ids:
            for ordinal in [0,1]:
                row = self.connection.execute('SELECT * FROM passages WHERE passage_id=?',(f'{article_id}:{ordinal}',)).fetchone()
                if row:
                    p=dict(row);candidates[p['passage_id']]=p
        ranked = []
        for p in candidates.values():
            text = p['text']
            # Sentence/line boundaries prevent a references list becoming context.
            for match in re.finditer(r'[^\n.!?]+(?:[.!?](?=\s|$)|$)',text):
                sentence = match.group().strip()
                words = WORD.findall(sentence)
                if len(words)<4:
                    continue
                normalized = {self.stem(w) for w in words}
                title_terms = {self.stem(w) for w in WORD.findall(p['title'])}
                coverage = len(query_terms & normalized)/len(query_terms)
                combined = len(query_terms & (normalized|title_terms))/len(query_terms)
                if combined < (0.6 if historical else 1) or coverage < (0.4 if historical else 0.5):
                    continue
                score = 6*coverage + 2*len(query_terms & title_terms)/len(query_terms)
                if re.search(r'\b(is|are|has|have|comprises|comprising|consists|contains)\b',sentence,re.I):score+=1.5
                if p['passage_id'].endswith(':0'):score+=1
                if sentence.casefold().startswith(p['title'].casefold()) and ' is ' in sentence.casefold():score+=1.5
                if exact_relation and exact_relation in sentence.casefold():score+=4
                if not historical and re.search(r'\b(was|were|former|formerly|until|before|historical|history|british raj)\b',sentence,re.I):score-=7
                if re.search(r'\b(references|external links|see also|list of|may refer to)\b',sentence,re.I):score-=5
                if 'capital' in query_terms and not historical:
                    if re.search(r'\b(financial|cultural|summer|economic|venture)\s+capital',sentence,re.I):score-=6
                    if re.search(r'\bnational capital\b',sentence,re.I):score+=2
                    modifier = re.search(r'\b(\w+) capital of\b',sentence,re.I)
                    if modifier and modifier.group(1).lower() not in {'the','national','is','as','and'}:score-=5
                    if re.search(r'\b(?:18|19)\d{2}\b',sentence):score-=4
                if count_noun:
                    if coverage < 1:score-=5
                    if re.search(r'\b(?:\d[\d,]*|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)\s+'+re.escape(count_noun)+r'\b',sentence,re.I):score+=8
                    else:score-=6
                if len(words)>65:score-=2
                focused = dict(p, text=sentence,original_text=text,
                    char_start=p['char_start']+match.start()+len(match.group())-len(match.group().lstrip()),
                    char_end=p['char_start']+match.end()-len(match.group())+len(match.group().rstrip()),
                    evidence_score=round(score,3))
                ranked.append((score,focused))
        ranked.sort(key=lambda pair:pair[0],reverse=True)
        selected=[];seen=set()
        for score,p in ranked:
            signature=' '.join(WORD.findall(p['text'].casefold()))
            if score < max(5,ranked[0][0]-4) or signature in seen or any(p['passage_id']==s['passage_id'] for s in selected):continue
            if count_noun and score<10:continue
            seen.add(signature);selected.append(p)
            if len(selected)==5:break
        diagnostics=dict(cached=False,candidate_passages=len(candidates),search_seconds=time.perf_counter()-start,
                         strategy='conjunction + optional relation phrase; article leads; sentence reranking')
        self.last_diagnostics=diagnostics
        if len(self.cache)>=100:self.cache.pop(next(iter(self.cache)))
        self.cache[key]=(selected,diagnostics)
        return copy.deepcopy(selected[:k])


In [6]:
class Generator:
    def __init__(self, model_dir: Path | None = None, device="cpu", threads=4,
                 max_input_tokens=512, max_new_tokens=24):
        offline_mode()
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        model_dir = model_dir or ROOT / "models" / "flan-t5-small"
        if not (model_dir / "model.safetensors").exists():
            raise FileNotFoundError(f"Model missing at {model_dir}. Run prepare while online.")
        if threads < 1 or max_new_tokens < 1 or not 64 <= max_input_tokens <= 512:
            raise ValueError("Require positive threads/output tokens and input budget 64..512.")
        torch.set_num_threads(threads)
        torch.manual_seed(42)
        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_dir, local_files_only=True, use_safetensors=True,
        ).to(device).eval()
        self.device = device
        self.max_input_tokens = max_input_tokens
        self.max_new_tokens = max_new_tokens
        self.threads = threads

    def count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=True, verbose=False))

    @staticmethod
    def render(question: str, evidence: list[dict], stage="answer") -> str:
        if stage == "closed_book":
            return f"Answer the question briefly.\nQuestion: {question}\nAnswer:"
        instruction = (
            "Copy the shortest phrase from the context that answers the question."
            if stage == "extract"
            else "Copy only the short answer phrase from the context. Do not repeat the context."
        )
        if re.search(r"\bhow many\b", question, re.I):
            instruction = "Answer with only the number requested, using the context."
        context = "\n\n".join(
            f'{item["title"]}\n{item["text"]}' for item in evidence
        )
        return (
            f"{instruction} If the context does not contain the answer, say unknown.\n"
            f"Context:\n{context}\nQuestion: {question}\nAnswer:"
        )

    def pack(self, question: str, evidence: list[dict], stage="answer"):
        """Fairly allocate a fixed input budget, preserving whole words and headers."""
        question = question.strip()
        if not question:
            raise ValueError("Question cannot be empty.")
        words = [re.findall(r"\S+", item["text"]) for item in evidence]

        def make(lengths):
            visible = [
                {key: item[key] for key in ("passage_id", "title", "url")}
                | {"text": " ".join(tokens[:length])}
                for item, tokens, length in zip(evidence, words, lengths)
            ]
            prompt = self.render(question, visible, stage)
            return prompt, visible, self.count_tokens(prompt)

        if not evidence:
            prompt, visible, size = make([])
            if size > self.max_input_tokens:
                raise ValueError("Question exceeds the model input budget.")
            return prompt, visible, size
        _, _, overhead = make([0] * len(words))
        if overhead > self.max_input_tokens - 16:
            raise ValueError("Question/source headers leave too little evidence space; reduce k.")
        low, high = 0, max(map(len, words))
        while low < high:
            middle = (low + high + 1) // 2
            if make([min(middle, len(tokens)) for tokens in words])[2] <= self.max_input_tokens:
                low = middle
            else:
                high = middle - 1
        lengths = [min(low, len(tokens)) for tokens in words]
        # Fill remaining capacity in retrieval order, never truncate the prompt.
        changed = True
        while changed:
            changed = False
            for i, tokens in enumerate(words):
                if lengths[i] < len(tokens):
                    trial = lengths.copy()
                    trial[i] += 1
                    if make(trial)[2] <= self.max_input_tokens:
                        lengths = trial
                        changed = True
        return make(lengths)

    def generate(self, prompt: str) -> dict:
        encoded = self.tokenizer(prompt, return_tensors="pt", truncation=False).to(self.device)
        input_tokens = int(encoded.input_ids.shape[1])
        if input_tokens > self.max_input_tokens:
            raise ValueError(f"Input has {input_tokens} tokens; budget is {self.max_input_tokens}.")
        start = time.perf_counter()
        with self.torch.inference_mode():
            output = self.model.generate(
                **encoded, do_sample=False, num_beams=1,
                max_new_tokens=self.max_new_tokens,
            )
        elapsed = time.perf_counter() - start
        return {
            "text": self.tokenizer.decode(output[0], skip_special_tokens=True).strip(),
            "input_tokens": input_tokens, "output_tokens": int(output.shape[1] - 1),
            "generation_seconds": elapsed,
        }


In [7]:
def supporting_sources(answer: str, visible: list[dict], originals: list[dict]) -> list[dict]:
    if is_unknown(answer):
        return []
    lookup = {item["passage_id"]: item for item in originals}
    sources = []
    for item in visible:
        original = lookup[item["passage_id"]]
        # Check both the model-visible excerpt and the original passage.
        if span(item["text"], answer) is not None:
            quote = span(original["text"], answer)
            if quote is not None:
                sources.append({
                    "passage_id": item["passage_id"], "title": original["title"],
                    "url": original["url"], "answer_span": quote,
                    "visible_excerpt": item["text"],
                    "char_start": original["char_start"], "char_end": original["char_end"],
                })
    return sources

def evidence_sentence(text: str, candidate: str) -> str:
    """Keep the candidate's original sentence so the reduce stage retains its relation."""
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        if span(sentence, candidate) is not None:
            return sentence.strip()
    # A generated span can cross sentence boundaries; retain the visible passage.
    return text


In [8]:
def answer_question(generator: Generator, question: str, retrieved: list[dict],
                    method="concatenate") -> dict:
    if method not in {"concatenate", "generate_from_evidence", "closed_book"}:
        raise ValueError(f"Unknown method: {method}")
    start = time.perf_counter()
    trace, map_steps = [], []
    if method == "closed_book":
        prompt, visible, _ = generator.pack(question, [], stage="closed_book")
        generated = generator.generate(prompt)
        trace.append({"stage": "closed_book", "prompt": prompt, **generated})
        raw_answer = generated["text"]
        sources = []
        answer = UNKNOWN if is_unknown(raw_answer) else raw_answer
    elif not retrieved:
        raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    else:
        evidence = retrieved
        if method == "generate_from_evidence":
            evidence = []
            for passage in retrieved:
                prompt, map_visible, _ = generator.pack(question, [passage], stage="extract")
                generated = generator.generate(prompt)
                trace.append({"stage": "extract", "prompt": prompt, **generated})
                candidate = generated["text"]
                supported = supporting_sources(candidate, map_visible, [passage])
                map_steps.append({
                    "passage_id": passage["passage_id"], "candidate": candidate,
                    "accepted": bool(supported), "visible_evidence": map_visible,
                })
                if supported:
                    # The reduce stage receives original quotes, never unverified model claims.
                    quote = evidence_sentence(map_visible[0]["text"], candidate)
                    map_steps[-1]["evidence_sentence"] = quote
                    evidence.append(passage | {"text": quote})
        if evidence:
            prompt, visible, _ = generator.pack(question, evidence)
            generated = generator.generate(prompt)
            trace.append({"stage": "answer", "prompt": prompt, **generated})
            raw_answer = generated["text"]
            sources = supporting_sources(raw_answer, visible, retrieved)
            # Short factual answers only; never return a generated paragraph as an answer.
            if len(WORD.findall(raw_answer)) > 16:
                sources = []
            if re.search(r"\bhow many\b",question,re.I) and not re.fullmatch(
                r"[\d,]+(?:\s+[A-Za-z]+){0,3}|(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)",raw_answer.strip(),re.I):
                sources = []
            count_question = re.search(r'how many\s+(.+?)(?:\s+(?:are|does|do|in|has|have|exist)\b|\?)',question.casefold()+'?')
            if count_question and sources:
                number = re.match(r'[\d,]+',raw_answer.strip())
                if number:
                    pattern = r'\b'+re.escape(number.group())+r'\s+'+re.escape(count_question.group(1))+r'\b'
                    sources = [source for source in sources if re.search(pattern,source['visible_excerpt'],re.I)]
            answer = raw_answer if sources else UNKNOWN
        else:
            raw_answer, answer, visible, sources = "", UNKNOWN, [], []
    return {
        "question": question, "method": method, "requested_passages_used": len(retrieved),
        "answer": answer, "raw_answer": raw_answer, "abstained": answer == UNKNOWN,
        "span_supported": bool(sources), "citations": sources, "retrieved": retrieved,
        "visible_evidence": visible, "map_steps": map_steps, "trace": trace,
        "generation_calls": len(trace),
        "total_input_tokens": sum(item["input_tokens"] for item in trace),
        "total_output_tokens": sum(item["output_tokens"] for item in trace),
        "latency_seconds": time.perf_counter() - start,
        "grounding_policy": "contiguous lexical span in visible and original evidence"
        if method != "closed_book" else "none (closed-book baseline)",
    }


In [9]:
# Close the old connection on rerun so revised retrieval code takes effect.
if 'ui_retriever' in globals():
    ui_retriever.close()
ui_retriever = Retriever(INDEX_PATH)
ui_generator = Generator(MODEL_DIR, device=DEVICE, threads=THREADS)
print('Ready: improved evidence retrieval and local Flan-T5-small loaded.')


W1025 10:25:48.111000 50780 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1025 10:25:48.244000 50780 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 10:25:48.271000 50780 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Ready: improved evidence retrieval and local Flan-T5-small loaded.


In [ ]:
def safe_link(url):
    return html.escape(url, quote=True) if urlparse(url).scheme in {'http','https'} else '#'

def render_result(result, retrieval_seconds):
    esc = html.escape
    answer = esc(result['answer'])
    status = 'No supported answer found' if result['abstained'] else 'Answer'
    parts = [f'<h3>{status}</h3><p style="font-size:1.15em">{answer}</p>']
    parts.append(f"<p>Retrieved {len(result['retrieved'])} passages in {retrieval_seconds:.1f}s · "
                 f"Answer pipeline: {result['latency_seconds']:.1f}s · "
                 f"Model calls: {result['generation_calls']}</p>")
    if result['abstained']:
        parts.append('<p>Try a more specific question or different retrieval size. Relevant information may still exist in Wikipedia.</p>')
    if result['citations']:
        parts.append('<h4>Source excerpts (evidence, not the answer)</h4>')
        for source in result['citations']:
            parts.append(f'<p><a href="{safe_link(source["url"])}" target="_blank" rel="noopener noreferrer">'
                         f'{esc(source["title"])}</a> — {esc(source["passage_id"])}</p>'
                         f'<blockquote>{esc(source["visible_excerpt"])}</blockquote>')
    parts.append('<details><summary>Inspect all retrieved passages</summary>')
    visible_ids = {p['passage_id'] for p in result['visible_evidence']}
    for rank,p in enumerate(result['retrieved'],1):
        parts.append(f'<h4>{rank}. {esc(p["title"])}</h4><p>{esc(p["text"])}</p>'
                     f'<small>Passage {esc(p["passage_id"])} · '
                     f'Included in final prompt: {p["passage_id"] in visible_ids}</small>')
    parts.append('</details><p><small>Source-span checks provide lexical support; they do not guarantee factual correctness.</small></p>')
    return ''.join(parts)

# Reexecuting this cell replaces the previous interface rather than leaving duplicate callbacks.
if 'qa_panel' in globals():
    qa_panel.close()
question_box = widgets.Textarea(placeholder='Example: What is the atomic number of aluminium?',
    description='Question:', layout=widgets.Layout(width='95%',height='85px'))
k_picker = widgets.Dropdown(options=[1,3,5],value=3,description='Max passages:')
method_picker = widgets.Dropdown(options=[('Concatenate (faster)','concatenate'),
    ('Generate from evidence','generate_from_evidence')],value='concatenate',description='Method:',
    layout=widgets.Layout(width='360px'))
ask_button = widgets.Button(description='Ask Wikipedia',button_style='primary',icon='search')
clear_button = widgets.Button(description='Clear')
status_label = widgets.HTML(value='Ready. Ask a factual question about the Wikipedia snapshot.')
answer_output = widgets.Output()
last_rag_result = None

def on_ask(_):
    global last_rag_result
    question = question_box.value.strip()
    if not question:
        status_label.value = 'Please enter a question.'
        return
    for control in [ask_button,clear_button,question_box,k_picker,method_picker]:
        control.disabled = True
    answer_output.clear_output(wait=True)
    status_label.value = '<b>Retrieving passages from the full Wikipedia index…</b>'
    try:
        start = time.perf_counter()
        passages_found = ui_retriever.search(question,k_picker.value)
        retrieval_seconds = time.perf_counter()-start
        status_label.value = '<b>Generating an answer from retrieved evidence…</b>'
        last_rag_result = answer_question(ui_generator,question,passages_found,method_picker.value)
        last_rag_result['retrieval_diagnostics'] = dict(ui_retriever.last_diagnostics)
        with answer_output:
            display(HTML(render_result(last_rag_result,retrieval_seconds)))
        status_label.value = '<b>Complete.</b> You can ask another question.'
    except Exception as exc:
        last_rag_result = None
        status_label.value = '<b>Could not complete the request.</b>'
        with answer_output:
            print(f'{type(exc).__name__}: {exc}')
    finally:
        for control in [ask_button,clear_button,question_box,k_picker,method_picker]:
            control.disabled = False

def on_clear(_):
    global last_rag_result
    question_box.value = ''
    last_rag_result = None
    answer_output.clear_output()
    status_label.value = 'Ready.'

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)
qa_panel = widgets.VBox([widgets.HTML('<h2>Ask Wikipedia</h2><p>Offline retrieval + Flan-T5-small · improved evidence ranking</p>'),
    question_box,widgets.HBox([k_picker,method_picker]),widgets.HBox([ask_button,clear_button]),
    status_label,answer_output])
display(qa_panel)
